# RSNA Knee Abnormality Detection — EDA

**Phase 2** of the standard DS workflow. Run after CSVs are available:

- Real data: `uv run bash scripts/download_csvs.sh`
- Local smoke data: `uv run python scripts/make_sample_data.py`
- Non-interactive summary: `uv run python scripts/run_eda.py` → `notes/eda_summary.md`

See also: `notes/problem_framing.md`

In [ ]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from config import TARGET_LABELS
from src.data import load_train, load_train_series
from src.labels import labeled_mask, add_weak_labels

sns.set_theme(style="whitegrid")

In [ ]:
train = load_train()
series = load_train_series()
print(f"Studies: {len(train):,}")
print(f"Series: {len(series):,}")
print(f"Series per study (median): {series.groupby('StudyInstanceUID').size().median():.0f}")
train.head()

In [ ]:
mask = labeled_mask(train)
print(f"Labeled studies: {mask.sum():,} / {len(train):,} ({100*mask.mean():.1f}%)")

label_counts = train.loc[mask, TARGET_LABELS].notna().sum().sort_values(ascending=False)
label_counts.plot(kind="barh", figsize=(8, 5), title="Expert labels available per class")
plt.tight_layout()

In [ ]:
train["report_len"] = train["Report"].fillna("").str.len()
train["report_len"].hist(bins=50, figsize=(8, 4))
plt.title("Report length (characters)")
plt.xlabel("chars")

In [ ]:
series["Anatomical_Plane"].value_counts().plot(kind="bar", figsize=(6, 3), title="Series by plane")
plt.xticks(rotation=0)

In [ ]:
# Compare weak keyword labels vs expert labels on labeled subset
weak_df = add_weak_labels(train.loc[mask].head(500))
weak_cols = [f"weak_{c}" for c in TARGET_LABELS]
agree = (weak_df[weak_cols].values == weak_df[TARGET_LABELS].fillna(-1).values).mean()
print(f"Keyword agreement with expert labels (sample): {agree:.1%}")

In [ ]:
# Positive rates among labeled studies (imbalance)
labeled = train.loc[mask]
rates = {}
for lab in TARGET_LABELS:
    col = labeled[lab]
    nn = col.notna()
    rates[lab] = (col[nn] == 1).mean() if nn.sum() else float("nan")
rate_s = pd.Series(rates).sort_values()
rate_s.plot(kind="barh", figsize=(8, 5), title="Positive rate among labeled (non-null)")
plt.xlabel("P(positive)")
plt.tight_layout()
print("Rarest:", rate_s.head(3).to_dict())
print("Most common:", rate_s.tail(3).to_dict())

In [ ]:
# Language / multilingual cues (heuristic)
reports = train["Report"].fillna("").astype(str)
print("Missing/empty reports:", int(((reports.str.strip() == "") | train["Report"].isna()).sum()))
for name, pat in {
    "spanish": r"\b(?:rodilla|informe|ligamento|menisco)\b",
    "french": r"\b(?:genou|compte rendu|ligament|ménisque|menisque)\b",
    "german": r"\b(?:knie|befund|bandruptur|meniskus)\b",
    "englishish": r"\b(?:mri|knee|tear|effusion|meniscus|acl)\b",
}.items():
    print(f"  {name} token hits: {reports.str.contains(pat, case=False, regex=True).sum()}")

from src.data import load_test
test = load_test()
print("Test has Report column:", "Report" in test.columns)
print("NOTE: Hidden test may omit reports — image model required for competitive score.")

In [ ]:
# Fluid-sensitive / fat-suppression breakdown
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
if "Fluid_Sensitive" in series.columns:
    series["Fluid_Sensitive"].value_counts().plot(kind="bar", ax=axes[0], title="Fluid_Sensitive")
if "Fat_Suppression" in series.columns:
    series["Fat_Suppression"].value_counts().plot(kind="bar", ax=axes[1], title="Fat_Suppression")
plt.tight_layout()

# Per-label weak vs expert agreement
weak_full = add_weak_labels(labeled.copy())
print("Per-label keyword agreement with expert:")
for lab in TARGET_LABELS:
    y = weak_full[lab]
    w = weak_full[f"weak_{lab}"]
    m = y.notna()
    if m.sum() == 0:
        continue
    print(f"  {lab:20s} {(y[m].astype(float) == w[m].astype(float)).mean():.1%}")

print("\nEDA complete. Write notes/eda_summary.md via: uv run python scripts/run_eda.py")